# 05 — Event Study: GOV Surprise vs. Abnormal Return

**Purpose**: Quantify the historical relationship between GOV surprise and
DASH abnormal return around earnings (CAR[-1,+2]).  
**Method**: CRSP-based abnormal returns (CRSP preferred; yfinance fallback).  
**Output**: Exhibit 6 scatter chart.  

See CLAUDE.md §13 for earnings dates and computation spec.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from src.config import CRSP_EVENT_STUDY_PATH, EARNINGS_DATES, CHART_STYLE, COLORS
from src.event_study import run_event_study, plot_event_study_scatter

plt.rcParams.update(CHART_STYLE)

## 1. Compute CARs for All Historical Earnings

In [ ]:
# Set q1_2026_surprise from GOV model forecast (notebook 02)
q1_2026_surprise_forecast = None  # fill after running notebook 02

event_df = run_event_study(q1_2026_surprise=q1_2026_surprise_forecast)
event_df

## 2. GOV Surprise → CAR Regression

In [ ]:
import statsmodels.api as sm

avail = event_df.dropna(subset=['gov_surprise_pct', 'car_minus1_plus2'])
if len(avail) >= 3:
    X = sm.add_constant(avail[['gov_surprise_pct']])
    y = avail['car_minus1_plus2'] * 100
    model = sm.OLS(y, X).fit()
    print(model.summary())
else:
    print('Insufficient observations for regression.')

## 3. Exhibit 6 — Event Study Scatter

In [ ]:
plot_event_study_scatter(event_df, q1_2026_surprise_forecast=q1_2026_surprise_forecast)

## 4. Interpretation for Write-Up

Template for write-up (Section 5, CLAUDE.md):

> *"A 1pp positive GOV surprise historically corresponds to a +Z% abnormal return 
> over the [-1,+2] event window (β={β}, R²={R²}, n={n} events). Applied to our 
> Q1 2026 model forecast of a +Xpp surprise vs. consensus, the implied CAR is 
> approximately +W%, providing a near-term catalyst validation signal."*